# Semana 2: Análisis de Ventas y Segmentación de Clientes

## Objetivos de la Clase

En esta sesión aprenderemos:

1. **Exploración rápida de datos**: Conocer nuestro dataset de ventas
2. **Manejo de fechas con .dt**: Extraer información temporal de las fechas
3. **Agregación con GroupBy**: Crear un dataframe de clientes con métricas clave
4. **Segmentación de clientes** usando:
   - `apply()`: Lógica condicional personalizada
   - `pd.cut()`: Segmentos con rangos definidos
   - `pd.qcut()`: Segmentos con cuantiles
   - **Análisis de Pareto**: Identificar el 80/20

---

## Dataset

Trabajaremos con un dataset de **ventas de bebidas** que contiene:
- Transacciones de clientes
- Productos comprados
- Fechas de compra
- Montos y cantidades

---
## Parte 1: Carga y Exploración de Datos

In [37]:
# Importar librerías
import pandas as pd
import numpy as np

In [ ]:
# Cargar datos desde: https://mysiterobert.s3.us-east-1.amazonaws.com/Dataframes/sales_drinks.csv
df = pd.read_csv("https://mysiterobert.s3.us-east-1.amazonaws.com/Dataframes/sales_drinks.csv")

In [ ]:
# Ver información del dataset
df.head(5)

,invoice_id,customer_id,purchase_datetime,sku,quantity,unit_price,discount_pct,amount,product_name,brand,category
0,100322793,966889,2026-04-08 17:41:58,43852960,3,12.00,0.0,36.00,Cerveza Pilsener Rubia Light 12 x 355 ml,Pilsener,beer_cider
1,100322793,966889,2026-04-08 17:41:58,41108238,2,5.35,0.0,10.70,Cerveza Club Premium Platino Lata Pack 6 x 355 ml,Club Premium,beer_cider
2,100159021,966889,2026-06-24 19:10:25,40217487,3,6.89,0.0,20.67,Cerveza Club Premium Clásica Botella (330 ml) ...,Club Premium,beer_cider
3,100159021,966889,2026-06-24 19:10:25,174161592,1,5.98,0.0,5.98,Cerveza Heineken Lata Pack (269 ml) 6 Unidades,Heineken,beer_cider
4,100159021,966889,2026-06-24 19:10:25,41108238,2,5.35,0.0,10.70,Cerveza Club Premium Platino Lata Pack 6 x 355 ml,Club Premium,beer_cider


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1834174 entries, 0 to 1834173
Data columns (total 11 columns):
 #   Column             Dtype  
---  ------             -----  
 0   invoice_id         int64  
 1   customer_id        int64  
 2   purchase_datetime  object 
 3   sku                int64  
 4   quantity           int64  
 5   unit_price         float64
 6   discount_pct       float64
 7   amount             float64
 8   product_name       object 
 9   brand              object 
 10  category           object 
dtypes: float64(3), int64(4), object(4)
memory usage: 153.9+ MB


In [ ]:
# Estadísticas descriptivas
df.describe()

,invoice_id,customer_id,sku,quantity,unit_price,discount_pct,amount
count,1.834174e+06,1.834174e+06,1.834174e+06,1.834174e+06,1.834174e+06,1.834174e+06,1.834174e+06
mean,1.001855e+08,5.501997e+05,8.403352e+07,6.935627e+00,7.568907e+00,2.884835e+00,4.051593e+01
std,1.070877e+05,2.589272e+05,6.142223e+07,1.324243e+01,4.827011e+00,6.035328e+00,8.477000e+01
min,1.000000e+08,1.000000e+05,4.021369e+07,1.000000e+00,2.500000e-01,0.000000e+00,2.500000e-01
25%,1.000926e+08,3.258380e+05,4.066886e+07,2.000000e+00,5.350000e+00,0.000000e+00,1.100000e+01
50%,1.001854e+08,5.502380e+05,4.110824e+07,3.000000e+00,5.350000e+00,0.000000e+00,1.941883e+01
75%,1.002783e+08,7.739740e+05,1.647673e+08,5.000000e+00,8.700000e+00,0.000000e+00,3.432000e+01
max,1.003710e+08,9.999820e+05,1.969781e+08,2.080000e+02,4.311000e+01,2.499986e+01,3.692548e+03


### Preguntas de exploración rápida

In [ ]:
# ¿Qué categorías existen?
df["category"].value_counts()

category
beer_cider                     1660442
soft_drinks_mixers               84083
wine_sparkling_wine              38955
spirits                          34947
juice_ice_tea_sports_energy       5833
water                             5098
pre_mixed                         4523
milk                               293
Name: count, dtype: int64

In [ ]:
# ¿Cuál ha sido el top 3 de categorías con mayores ventas?
df.groupby("category")["amount"].sum().sort_values(ascending= False).head(3)

category
beer_cider            6.468051e+07
soft_drinks_mixers    6.263647e+06
spirits               1.918637e+06
Name: amount, dtype: float64

In [ ]:
# ¿Cuántos clientes únicos hay?
df["customer_id"].nunique()

53000

---
## Parte 2: Manejo de Fechas con .dt

El accessor `.dt` nos permite acceder a propiedades de fechas cuando una columna es tipo datetime.

In [ ]:
# Convertir la columna 'purchase_datetime' a tipo datetime
df["purchase_datetime"] = pd.to_datetime(df["purchase_datetime"])

### Extraer componentes de la fecha

In [ ]:
print(df["purchase_datetime"].min())
print(df["purchase_datetime"].max())


2026-03-07 00:01:39
2026-09-03 23:59:59


In [ ]:
# Extraer año, mes, día
# Extraer día de la semana (0=Lunes, 6=Domingo)
# Nombre del día de la semana
# Nombre del mes

In [ ]:
df["dia"] = df["purchase_datetime"].dt.day
df["month"] = df["purchase_datetime"].dt.month
df["year"] = df["purchase_datetime"].dt.year
df["week_day"] = df["purchase_datetime"].dt.weekday
df["week_day_name"] = df["purchase_datetime"].dt.day_name()

In [ ]:
#dir("")

In [ ]:
# ¿Cuántas ventas (facturas únicas) hubo en cada mes?
display(df.groupby("month")["invoice_id"].nunique())
print("Clientes")
display(df.groupby("month")["customer_id"].nunique())

month
3    48933
4    56365
5    59710
6    59261
7    65850
8    73371
9     7510
Name: invoice_id, dtype: int64

Clientes


month
3    25352
4    27605
5    28434
6    28230
7    29398
8    30111
9     5940
Name: customer_id, dtype: int64

In [ ]:
# ¿Qué día de la semana se vende más? 
# ventas...
# unidades vendidas
df.groupby("week_day")["amount"].sum().sort_values(ascending=False)

week_day
1    1.203889e+07
2    1.197852e+07
0    1.178890e+07
3    9.933724e+06
6    9.683081e+06
5    9.617668e+06
4    9.272487e+06
Name: amount, dtype: float64

In [ ]:
df.groupby("week_day_name")[["amount", "quantity"]].sum().sort_values(by = ["amount"], ascending=False)

,amount,quantity
week_day_name,,
Tuesday,1.203889e+07,2065434
Wednesday,1.197852e+07,2045733
Monday,1.178890e+07,2013497
Thursday,9.933724e+06,1700597
Sunday,9.683081e+06,1656366
Saturday,9.617668e+06,1637307
Friday,9.272487e+06,1602213


---
## Parte 3: Agregación - Creando el DataFrame de Clientes

Vamos a crear un nuevo DataFrame (`df_clients`) con métricas importantes por cliente.

In [ ]:
df
# ticket promedio
# numero de facturas
# producto más comprado
# frecuencia de compra
# día de mayor compra
# monto total comprado
# descuento promedio
# sucursal más frecuente
# categoría más comprado
# primera vez que compró
# marca favoria
# última compra 
# cuantos días ha pasado de la última compra
# horario de compra (segmento)
# portafolio de productos (cantidad de productos únicos) (Sku x poc)


,invoice_id,customer_id,purchase_datetime,sku,quantity,unit_price,discount_pct,amount,product_name,brand,category,dia,month,year,week_day,week_day_name
0,100322793,966889,2026-04-08 17:41:58,43852960,3,12.00,0.0,36.00,Cerveza Pilsener Rubia Light 12 x 355 ml,Pilsener,beer_cider,8,4,2026,2,Wednesday
1,100322793,966889,2026-04-08 17:41:58,41108238,2,5.35,0.0,10.70,Cerveza Club Premium Platino Lata Pack 6 x 355 ml,Club Premium,beer_cider,8,4,2026,2,Wednesday
2,100159021,966889,2026-06-24 19:10:25,40217487,3,6.89,0.0,20.67,Cerveza Club Premium Clásica Botella (330 ml) ...,Club Premium,beer_cider,24,6,2026,2,Wednesday
3,100159021,966889,2026-06-24 19:10:25,174161592,1,5.98,0.0,5.98,Cerveza Heineken Lata Pack (269 ml) 6 Unidades,Heineken,beer_cider,24,6,2026,2,Wednesday
4,100159021,966889,2026-06-24 19:10:25,41108238,2,5.35,0.0,10.70,Cerveza Club Premium Platino Lata Pack 6 x 355 ml,Club Premium,beer_cider,24,6,2026,2,Wednesday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1834169,100215440,796558,2026-05-09 14:39:51,164767306,3,6.18,0.0,18.54,Cerveza Club Premium Clásica Lata Pack 6 x 269 ml,Club Premium,beer_cider,9,5,2026,5,Saturday
1834170,100215440,796558,2026-05-09 14:39:51,40668863,3,8.70,0.0,26.10,Cerveza Pilsener Lata Pack 12 x 269 ml,Pilsener,beer_cider,9,5,2026,5,Saturday
1834171,100005323,180325,2026-07-10 05:00:36,164767306,3,6.18,0.0,18.54,Cerveza Club Premium Clásica Lata Pack 6 x 269 ml,Club Premium,beer_cider,10,7,2026,4,Friday
1834172,100005323,180325,2026-07-10 05:00:36,41108238,2,5.35,0.0,10.70,Cerveza Club Premium Platino Lata Pack 6 x 355 ml,Club Premium,beer_cider,10,7,2026,4,Friday


In [ ]:
# Crear dataframe de clientes con métricas agregadas:
# - num_compras: Número de facturas únicas
# - num_skus: Número de SKUs únicos comprados
# - total_comprado: Total gastado
# - ultima_compra: Fecha de última compra
# - num_meses_comprados: Número de meses únicos en los que compró
# Calcular ticket promedio (total_comprado / num_compras) OJO

In [ ]:
df.columns

Index(['invoice_id', 'customer_id', 'purchase_datetime', 'sku', 'quantity',
       'unit_price', 'discount_pct', 'amount', 'product_name', 'brand',
       'category', 'dia', 'month', 'year', 'week_day', 'week_day_name'],
      dtype='object')

In [ ]:
df_clients = df.groupby("customer_id").agg({
    "invoice_id": "nunique", 
    "sku": "nunique", 
    "amount": "sum", 
    "purchase_datetime": "max", 
    "month": "nunique"
})

df_clients["ticket_promedio"] = df_clients["amount"] / df_clients["invoice_id"]
df_clients.head(2)

,invoice_id,sku,amount,purchase_datetime,month,ticket_promedio
customer_id,,,,,,
100000,3,7,183.754259,2026-07-22 08:11:49,3,61.251420
100002,16,17,1490.208144,2026-08-05 18:02:39,6,93.138009


In [ ]:
df_clients.columns = ["num_facturas", "skus", "monto_total", "ultima_compra", "num_meses", "ticket_promedio"]

In [ ]:
df_clients = df_clients.reset_index()

In [ ]:
# Calcular días desde última compra
# Usa la fecha máxima del dataset como referencia (hoy)

In [ ]:
# Ver resumen de métricas de clientes
df_clients.describe()

# ticket promedio = AOV (Average order value) = Cuando compra el cliente cada que vez que compra

,customer_id,num_facturas,skus,monto_total,ultima_compra,num_meses,ticket_promedio
count,53000.000000,53000.000000,53000.000000,53000.000000,53000,53000.000000,53000.000000
mean,549898.912226,7.000000,10.342094,1402.137234,2026-07-27 21:00:30.030037760,3.303208,134.773834
min,100000.000000,1.000000,1.000000,2.800000,2026-03-07 11:20:12,1.000000,2.800000
25%,325083.000000,2.000000,6.000000,242.294763,2026-07-10 06:51:01.500000,2.000000,86.549862
50%,549591.500000,4.000000,9.000000,486.932244,2026-08-10 17:13:33.500000,3.000000,116.623687
75%,774975.500000,7.000000,13.000000,894.368918,2026-08-26 11:11:18.500000,4.000000,154.433376
max,999982.000000,134.000000,46.000000,102984.585221,2026-09-03 23:59:59,7.000000,1239.037344
std,259752.387492,9.954653,6.357003,5480.941961,NaN,1.629586,98.898733


---
## Parte 4: Segmentación de Clientes

Ahora vamos a crear diferentes segmentaciones usando distintas técnicas.

### Segmento 1: Usando Apply con Lógica Simple

Clasificar clientes según si compraron más o menos de $1000.

In [ ]:
# Crear una función que clasifique:
# - '>$1000' si total > 1000
# - '<=$1000' en caso contrario

# Aplicar la función a la columna 'total_comprado' y crear columna 'segmento_1'
def segmento_1(monto):
    if monto > 1000:
        return ">$1000"
    return "<=$1000"

df_clients["segmento_1"] = df_clients["monto_total"].apply(segmento_1)
# Contar clientes por segmento


In [ ]:
df_clients["segmento_1"] = df_clients["monto_total"].apply(lambda monto: ">$1000" if (monto>1000) else "<=$1000")

### Segmento 2: Usando pd.cut() - Rangos Definidos

`pd.cut()` nos permite crear segmentos definiendo los límites de los rangos.

In [ ]:
# Definir los límites de los rangos: 0, 100, 200, 500, infinito
# Definir las etiquetas: '<=100', '(100-200]', '(200-500]', '>500'

# Usar pd.cut() para crear la columna 'segmento_2'

# Contar clientes por segmento
def segmento_2(monto):
    if monto<=100:
        return '<=$100'
    if monto <= 200:
        return '($100-$200]'
    if monto <=500:
        return '($200-$500]'
    return ">$500"

df_clients["segmento_2"] = df_clients["monto_total"].apply(segmento_2)

### Segmento 3: Portafolio - Usando Apply con Lógica Compleja

In [ ]:
# Crear una función que clasifique por cantidad de SKUs:
# - 'Monoproducto' si num_skus == 1
# - 'Portafolio pequeño' si num_skus <= 3
# - 'Portafolio mediano' si num_skus <= 8
# - 'Portafolio amplio' si num_skus > 8

# Aplicar la función y crear columna 'segmento_portafolio'
def segmento_portafolio(s):
    if s == 1:
        return "Monoproducto"
    if s <= 3: 
        return "Portafolio pequeño"
    if s<= 8:
        return "Portafolio grande"
    return "Portafolio amplio"

# Contar clientes por segmento


### Segmento 4: Análisis de Pareto (80/20)

El principio de Pareto dice que el 80% de las ventas viene del 20% de los clientes.

**Objetivo**: Identificar qué clientes generan el 80% de las ventas totales.

**Funciones importantes para este ejercicio**:
- `.sort_values()`: Ordena un DataFrame por una o más columnas
- `.cumsum()`: Calcula la suma acumulada (running total)
- `np.where()` o `.apply()`: Para crear columnas condicionales

**Pasos a seguir**:

1. **Ordenar** el DataFrame `df_clients` por `monto_total` de mayor a menor
2. **Calcular** el total de ventas de TODOS los clientes (suma de `monto_total`)
3. **Calcular** las ventas acumuladas usando `.cumsum()` en la columna `monto_total`
4. **Calcular** el porcentaje acumulado: (ventas_acumuladas / total_ventas) * 100
5. **Identificar** clientes Pareto: aquellos donde el porcentaje acumulado es <= 80
6. **Crear** una columna llamada `segmento_pareto` que diga 'Pareto' o 'No Pareto'

<details>
<summary>💡 Hint 1: ¿Cómo ordenar de mayor a menor?</summary>

```python
# Para ordenar de mayor a menor usa ascending=False
# IMPORTANTE: Puedes asignar el resultado a la misma variable para modificar df_clients
df_clients = df_clients.sort_values(by='monto_total', ascending=False)
```
</details>

<details>
<summary>💡 Hint 2: ¿Cómo calcular el total?</summary>

```python
# Usa .sum() para obtener la suma total de una columna
total_ventas = df_clients['monto_total'].sum()
```
</details>

<details>
<summary>💡 Hint 3: ¿Cómo calcular porcentaje acumulado?</summary>

```python
# Primero calcula la suma acumulada directamente en df_clients
df_clients['monto_acumulado'] = df_clients['monto_total'].cumsum()

# Luego divide entre el total y multiplica por 100
df_clients['pct_acumulado'] = (df_clients['monto_acumulado'] / total_ventas) * 100
```
</details>

<details>
<summary>💡 Hint 4: ¿Cómo crear la columna segmento_pareto?</summary>

```python
# Usa np.where() o un apply con lambda
df_clients['segmento_pareto'] = np.where(df_clients['pct_acumulado'] <= 80, 'Pareto', 'No Pareto')

# O con apply:
df_clients['segmento_pareto'] = df_clients['pct_acumulado'].apply(lambda x: 'Pareto' if x <= 80 else 'No Pareto')
```
</details>

<details>
<summary>💡 Hint 5: Estructura completa del ejercicio</summary>

```python
# 1. Ordenar (modifica df_clients directamente)
df_clients = df_clients.sort_values(by='monto_total', ascending=False)

# 2. Total de ventas
total_ventas = df_clients['monto_total'].sum()

# 3. Suma acumulada (crea columna en df_clients)
df_clients['monto_acumulado'] = df_clients['monto_total'].cumsum()

# 4. Porcentaje acumulado (crea columna en df_clients)
df_clients['pct_acumulado'] = (df_clients['monto_acumulado'] / total_ventas) * 100

# 5. Crear columna segmento_pareto
df_clients['segmento_pareto'] = np.where(df_clients['pct_acumulado'] <= 80, 'Pareto', 'No Pareto')

# 6. Contar cuántos clientes hay en cada segmento
df_clients['segmento_pareto'].value_counts()
```
</details>

In [ ]:
# EJERCICIO: Análisis de Pareto
# Completa el código siguiendo los pasos descritos arriba

# Paso 1: Ordenar clientes por monto_total (descendente)


# Paso 2: Calcular el total de ventas


# Paso 3: Calcular las ventas acumuladas (cumsum)


# Paso 4: Calcular el porcentaje acumulado


# Paso 5: Crear columna 'segmento_pareto' (Pareto si pct_acumulado <= 80)


# Paso 6: Mostrar resultados
# ¿Cuántos clientes son Pareto vs No Pareto?


# ¿Qué porcentaje de clientes son Pareto?


### Segmento 5: Usando pd.qcut() - Cuantiles

`pd.qcut()` divide los datos en grupos de igual **cantidad** de observaciones (cuantiles).

**¿Cuál es la diferencia entre `pd.cut()` y `pd.qcut()`?**
- `pd.cut()`: Divide por **rangos de valores** (ej: 0-100, 100-200, etc.)
- `pd.qcut()`: Divide por **cantidad de datos** (ej: cada grupo tiene ~25% de los datos si usas 4 cuartiles)

**Función importante**:
```python
pd.qcut(x, q, labels=None)
```
- `x`: La columna/Serie a segmentar
- `q`: Número de cuantiles (ej: 4 para cuartiles)
- `labels`: Etiquetas personalizadas para cada grupo

**Objetivo**: Dividir a los clientes en 4 grupos (A, B, C, D) según su `monto_total`:
- **A**: Top 25% (los que más compran)
- **B**: Siguiente 25%
- **C**: Siguiente 25%
- **D**: Bottom 25% (los que menos compran)

<details>
<summary>💡 Hint 1: Sintaxis básica de pd.qcut()</summary>

```python
# Para dividir en 4 grupos iguales (cuartiles)
df['segmento'] = pd.qcut(df['columna'], q=4, labels=['D', 'C', 'B', 'A'])

# Nota: Las etiquetas van de menor a mayor valor
# Como queremos que A sea el top, ponemos las etiquetas al revés
```
</details>

<details>
<summary>💡 Hint 2: ¿Cómo contar clientes por segmento?</summary>

```python
# Usa .value_counts() para contar
df['segmento_abcd'].value_counts()

# O usa .sort_index() para ordenar alfabéticamente
df['segmento_abcd'].value_counts().sort_index()
```
</details>

<details>
<summary>💡 Hint 3: ¿Cómo ver estadísticas por segmento?</summary>

```python
# Usa .groupby() con .describe()
df.groupby('segmento_abcd')['monto_total'].describe()

# O calcula métricas específicas
df.groupby('segmento_abcd')['monto_total'].agg(['count', 'mean', 'min', 'max'])
```
</details>

<details>
<summary>💡 Hint 4: Orden de las etiquetas</summary>

Recuerda que las etiquetas en `pd.qcut()` se asignan de **menor a mayor** valor:
- Primera etiqueta → valores más bajos
- Última etiqueta → valores más altos

Si quieres que **A** represente los que **más** compran, las etiquetas deben ser: `['D', 'C', 'B', 'A']`
</details>

In [ ]:
# EJERCICIO: Segmentación con pd.qcut()
# Completa el código siguiendo las instrucciones

# Segmentar en 4 grupos (cuartiles) según monto_total
# A = los que más compran, D = los que menos compran
# Crear columna 'segmento_abcd'


# Contar clientes por segmento


# Ver estadísticas por segmento (count, mean, min, max)


# PREGUNTA DE ANÁLISIS:
# ¿Cuál es el monto mínimo que debe gastar un cliente para estar en el segmento A?


---
## Parte 5: Análisis de Segmentos

Ahora que tenemos todos los segmentos, hagamos algunos análisis para entender mejor a nuestros clientes.

**Funciones importantes para esta sección**:
- `.groupby()`: Agrupa datos por una o más columnas
- `.agg()`: Aplica múltiples funciones de agregación
- `pd.crosstab()`: Crea tablas de frecuencia cruzadas (como una tabla dinámica)

### EJERCICIO 1: Explorar las columnas de segmentación

Muestra las primeras 10 filas con SOLO las columnas de segmentación:
- `customer_id`
- `segmento_1`
- `segmento_2`
- `segmento_portafolio`
- `segmento_pareto`
- `segmento_abcd`

<details>
<summary>💡 Hint: Seleccionar columnas específicas</summary>

```python
# Para seleccionar columnas específicas usa una lista
df[['columna1', 'columna2', 'columna3']].head(10)
```
</details>

In [ ]:
# Tu código aquí:


### EJERCICIO 2: Comparar segmentos Pareto vs No Pareto

**Objetivo**: Comparar las métricas entre clientes Pareto y No Pareto.

**Métricas a mostrar**:
- `count`: Cantidad de clientes
- `sum` de `monto_total`: Total vendido
- `mean` de `monto_total`: Promedio de compra
- `mean` de `num_facturas`: Promedio de compras
- `mean` de `ticket_promedio`: Ticket promedio

<details>
<summary>💡 Hint 1: ¿Cómo usar .groupby() con .agg()?</summary>

```python
# Sintaxis de .agg() con múltiples columnas y funciones
df.groupby('columna_segmento').agg({
    'columna1': 'funcion1',
    'columna2': ['funcion1', 'funcion2'],
    'columna3': 'funcion3'
})
```
</details>

<details>
<summary>💡 Hint 2: Estructura del ejercicio</summary>

```python
# Agrupa por segmento_pareto y aplica agregaciones
resultado = df_clients.groupby('segmento_pareto').agg({
    'customer_id': 'count',  # Cantidad de clientes
    'monto_total': ['sum', 'mean'],  # Total y promedio
    'num_facturas': 'mean',  # Promedio de facturas
    'ticket_promedio': 'mean'  # Ticket promedio
})
```
</details>

### EJERCICIO 3: Análisis cruzado - Segmento ABCD vs Portafolio

**Objetivo**: Crear una tabla que muestre cuántos clientes hay en cada combinación de segmento ABCD y segmento de portafolio.

**Función a usar**: `pd.crosstab()`

```python
pd.crosstab(index, columns)
```
- `index`: La variable para las filas
- `columns`: La variable para las columnas

**Ejemplo de resultado esperado**:
```
segmento_portafolio   Monoproducto  Portafolio pequeño  ...
segmento_abcd                                            
A                            500                 1200    ...
B                           1000                 2000    ...
...
```

<details>
<summary>💡 Hint 1: Sintaxis de pd.crosstab()</summary>

```python
# Para crear una tabla cruzada entre dos variables
tabla = pd.crosstab(df['variable_filas'], df['variable_columnas'])
```
</details>

<details>
<summary>💡 Hint 2: ¿Cómo interpretar el resultado?</summary>

La tabla te mostrará:
- **Filas**: Los segmentos ABCD
- **Columnas**: Los tipos de portafolio
- **Valores**: Cantidad de clientes en esa intersección

Por ejemplo, si ves "500" en la celda (A, Monoproducto), significa que hay 500 clientes que están en el segmento A y compran un solo producto.
</details>

<details>
<summary>💡 Hint 3: Agregar totales</summary>

```python
# Para agregar totales por fila y columna
pd.crosstab(df['var1'], df['var2'], margins=True)
```
</details>

### EJERCICIO 4: ¿Los clientes Pareto tienen portafolios más amplios?

**Objetivo**: Calcular el promedio de SKUs únicos (`skus`) para cada segmento Pareto.

**Pasos**:
1. Agrupa por `segmento_pareto`
2. Calcula el promedio (`.mean()`) de la columna `skus`

<details>
<summary>💡 Hint: Sintaxis</summary>

```python
# Opción 1: Usando .groupby() con indexación
df.groupby('columna_grupo')['columna_a_promediar'].mean()

# Opción 2: Usando .groupby() con .agg()
df.groupby('columna_grupo').agg({'columna_a_promediar': 'mean'})
```
</details>

In [ ]:
# Tu código aquí:


# PREGUNTA DE ANÁLISIS:
# ¿Los clientes Pareto compran más variedad de productos que los No Pareto?
# Escribe tu conclusión aquí como comentario
